# Constrain piece-wise constant pulses to vary smoothly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.measurement.smoothness import Smoothness
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.scipy_optimizer_gradient import ScipyOptimizerGradient

In [ ]:
delta_sampling = 33e-9
n_pwc = 40  # number of piecewise constants in the pulse
t_final = n_pwc * delta_sampling  # for now just set up for trying
tlist = np.linspace(0, t_final, n_pwc + 1)
eps_qubit = 2 * np.pi * 1.0  # initial amplitude of the qubit (in MHz)
eps_max_qubit = 5 * eps_qubit  # maximum amplitude of the resonator (in MHz)
tone_qubit = GaussEnvelope(
    amplitude=Quantity(eps_qubit * 1e6, -eps_max_qubit * 1e6, eps_max_qubit * 1e6),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final),
)
gen_qubit = PWCGenerator(envelopes=[tone_qubit], max_amplitude=eps_max_qubit * 1e6, tlist=tlist)

In [ ]:
def plot_pulse(ts: np.ndarray, tone_qubit: GaussEnvelope):
    """Function to plot the pulse."""
    _, ax = plt.subplots(1, figsize=(5, 3), sharex=True)
    ax.plot(ts / 1e-9, tone_qubit.get_value(ts) / 1e6 / 2 * np.pi, label="Initial smooth curve")
    ax.plot(ts / 1e-9, np.real(gen_qubit.get_value(ts)) / 1e6 / 2 * np.pi, ls="--", label="in-phase")
    ax.plot(
        ts / 1e-9,
        np.imag(gen_qubit.get_value(ts)) / 1e6,
        ls="--",
        label="out-of-phase",
    )

    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Amplitude (MHz)")
    ax.legend()
    ax.grid()
    plt.show()


# plot_pulse(ts, tone_qubit)

In [ ]:
optmap = OptimizationMap()
optmap.add(gen_qubit, gen_qubit.get_parameters())

# We add dummy parameters to check if the gradient is computed
# correctly by padding zeros
# optmap.add(tone_qubit, tone_qubit.get_parameters())
optmap.register_params_with_optimizables()

In [ ]:
smoothness = Smoothness(pwc_generator=gen_qubit)

We can check that the gradient has the correct shape

In [ ]:
print(smoothness.get_value_and_gradient(tlist)[1].shape)

In [ ]:
opt = ScipyOptimizerGradient(smoothness, optimization_map=optmap)
max_iter = 200
opt.set_options({"maxiter": max_iter})

In [ ]:
# opt.optimize(ts)

In [ ]:
# smoothness.measure(ts)

In [ ]:
# smoothness.get_value_and_gradient(ts)

In [ ]:
# plot_signal(gen_qubit, ts, linestyle="--", label="PWC");

As expected obtain a flat pulse.